* Install additional package and requirements for this bonus notebook :

In [ ]:
!pip install -r requirements-extra.txt

## Comparing Various Byte Pair Encoding (BPE) Implementations ...



### Using BPE from `tiktoken`

In [ ]:
from importlib.metadata import version

print("tiktoken version", version("tiktoken"))

tiktoken version 0.12.0


In [ ]:
import tiktoken

tik_tokenizer = tiktoken.get_encoding("gpt2")

text = "Hello , world. Is this-- a test?"

In [ ]:
integers = tik_tokenizer.encode(text, allowed_special={"<|endodtext|>"})

print(integers)

[15496, 837, 995, 13, 1148, 428, 438, 257, 1332, 30]


In [ ]:
strings = tik_tokenizer.decode(integers)

print(strings)

Hello , world. Is this-- a test?


In [ ]:
print(tik_tokenizer.n_vocab)

50257


### Using the original BPE implementation used in GPT-2

In [ ]:
from bpe_openai_gpt2 import get_encoder, download_vocab

download_vocab()


Fetching encoder.json: 100%|███████████████████████████████████| 1.04M/1.04M [00:00<00:00, 5.12MB/s]
Fetching vocab.bpe: 100%|████████████████████████████████████████| 456k/456k [00:00<00:00, 3.29MB/s]


True

In [ ]:
orig_tokenizer = get_encoder(model_name="gpt2_model", models_dir=".")


In [ ]:
integers = orig_tokenizer.encode(text)

print(integers)


[15496, 837, 995, 13, 1148, 428, 438, 257, 1332, 30]


In [ ]:
strings = orig_tokenizer.decode(integers)
print(strings)

Hello , world. Is this-- a test?


## Using BPE via Hugging Face transformers


In [ ]:
import transformers

transformers.__version__

'4.57.3'

In [ ]:
from transformers import GPT2Tokenizer

hf_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [ ]:
hf_tokenizer(strings)["input_ids"]

[15496, 837, 995, 13, 1148, 428, 438, 257, 1332, 30]

In [ ]:
from transformers import GPT2TokenizerFast

hf_tokenizer_fast = GPT2TokenizerFast.from_pretrained("gpt2")

In [ ]:
hf_tokenizer_fast(strings)["input_ids"]

[15496, 837, 995, 13, 1148, 428, 438, 257, 1332, 30]

##Using Our own from-scratch BPE tokenizer

In [ ]:
# import os
# import sys
# import io
# import nbformat
# import types

# def import_from_notebook():
#   def import_definitions_from_notebook(fullname, names):
#     current_dir = os.getcwd()
#     path = os.path.join(current_dir, "..", "05_bpe-from-scratch", fullname + ".ipynb")
#     path = os.path.normpath(path)

#     # Load the notebook
#     if not os.path.exists(path):
#       raise FileNotFoundError(f"Notebook file is not found at: {path}")

#     with io.open(path , "r", encoding="utf-8") as f:
#       nb = nbformat.read(f, as_version=4)

#     # Create a module to store the imported functions and classes.
#     mod = types.ModuleType(fullname)
#     sys.modules[fullname] = mod

#     # Go through the notebook cells and execute function or class definitions
#     for cell in nb.cells:
#       if cell.cell_type == "code":
#         cell_code = cell.source
#         for name in names:
#           # Check for function or class definitions
#           if f"def {name}" in cell_code or f"class {name}" in cell_code:
#             exec(cell_code, mode.__dict__)
#     return mod
#   fullname = "bpe-from-scrach"
#   names = ["BPETokenizerSimple"]

#   return import_definitions_from_notebook(fullname, names)




In [ ]:

import os
import sys
import io
import nbformat
import types

def import_from_notebook():
    def import_definitions_from_notebook(fullname, names):
        current_dir = os.getcwd()
        path = os.path.join(current_dir, "..", "05_bpe-from-scratch", fullname + ".ipynb")
        path = os.path.normpath(path)

        # Load the notebook
        if not os.path.exists(path):
            raise FileNotFoundError(f"Notebook file not found at: {path}")

        with io.open(path, "r", encoding="utf-8") as f:
            nb = nbformat.read(f, as_version=4)

        # Create a module to store the imported functions and classes
        mod = types.ModuleType(fullname)
        sys.modules[fullname] = mod

        # Go through the notebook cells and only execute function or class definitions
        for cell in nb.cells:
            if cell.cell_type == "code":
                cell_code = cell.source
                for name in names:
                    # Check for function or class definitions
                    if f"def {name}" in cell_code or f"class {name}" in cell_code:
                        exec(cell_code, mod.__dict__)
        return mod

    fullname = "bpe-from-scratch"
    names = ["BPETokenizerSimple"]

    return import_definitions_from_notebook(fullname, names)

In [ ]:
imported_module = import_from_notebook()

BPETokenizerSimple = getattr(imported_module, "BPETokenizerSimple", None)

tokenizer_gpt2 = BPETokenizerSimple()
tokenizer_gpt2.load_vocab_and_merges_from_openai(
    vocab_path=os.path.join("gpt2_model", "encoder.json"),
    bpe_merges_path=os.path.join("gpt2_model", "vocab.bpe")
)

FileNotFoundError: Notebook file not found at: /05_bpe-from-scratch/bpe-from-scratch.ipynb

In [ ]:
integers = tokenizer_gpt2.encode(text)
print(integers)

NameError: name 'tokenizer_gpt2' is not defined

## A quick performance BenchMark


In [ ]:
from google.colab import files
uploaded = files.upload()


with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()


Saving the-verdict.txt to the-verdict.txt


### Note:
**The change in the below code is becuase we worked with different stucture in colab and in github we followed the correct structre, so the file is opened from the downloads than from the first chapter folder**

In [ ]:
# with open("../Main_Chapter_code/the-verdict.txt", "r", encoding="utf-8") as f:
#     raw_text = f.read()


with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()


## Original OpenAI GPT-2 tokenizer

In [ ]:
%timeit orig_tokenizer.encode(raw_text)

13.6 ms ± 2.75 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [38]:
## TikToken OpenAI GPT-2 Tokenizer

%timeit tik_tokenizer.encode(raw_text)


3.4 ms ± 1.13 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


## Hugging Face openAI GPT-2-tokenizer

In [40]:
%timeit hf_tokenizer(raw_text)["input_ids"]

231 ms ± 65.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [41]:
%timeit hf_tokenizer(raw_text, max_length=5125, truncation=True)["input_ids"]

37.8 ms ± 10.3 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [42]:
%timeit hf_tokenizer_fast(raw_text)["input_ids"]

Token indices sequence length is longer than the specified maximum sequence length for this model (5145 > 1024). Running this sequence through the model will result in indexing errors


17.3 ms ± 3.62 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [43]:
%timeit hf_tokenizer_fast(raw_text, max_length=5145, truncation=True)["input_ids"]

16.7 ms ± 3.45 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


## Our Own GPT-Tokenizer (For Educational Purpose)


In [44]:
%timeit tokenizer_gpt2.encode(raw_text)

NameError: name 'tokenizer_gpt2' is not defined